# validation-no-grad — ex1: validation pass wrapped in torch.no_grad

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `validation-no-grad`. Running the final beacon cell reports progress against the `PyTorch: no_grad validation` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: no_grad validation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`validation-no-grad`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "validation-no-grad"
DD_SUBTOPIC = "PyTorch: no_grad validation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Validation under `torch.no_grad()` — quick refresher

During validation/inference you don't need gradients — and computing them anyway wastes memory (the autograd graph is kept alive) and time (every op records its backward closure). `torch.no_grad()` is a context manager (or decorator) that disables gradient tracking for the code inside it. `torch.inference_mode()` is the newer, even-stricter version that additionally disables version counters.

**Tensors produced inside `no_grad` have `requires_grad=False` and no `grad_fn`.** Calling `.backward()` on them raises. Calling `.item()` or `.numpy()` is fine.

**Always pair with `model.eval()`.** `no_grad` controls autograd; `model.eval()` controls layer behavior (BatchNorm / Dropout). They are independent — you need both for honest validation.

### Exercise 1 — validation pass wrapped in torch.no_grad

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `torch.no_grad()` as a context manager around an inference pass so that produced tensors have no `grad_fn` and `.backward()` cannot be called on them.
> Keywords: validation, no-grad, inference
> ```

**KCs targeted:** `no-grad-context-manager-usage`, `no-grad-disables-grad-fn`

Implement `ex1_validate(model, x, y)`. A textbook validation pass.

1. Open a `with t.no_grad():` block.
2. Inside, run `logits = model(x)`.
3. Inside, compute `loss = ((logits - y) ** 2).mean()`.
4. Inside, extract `loss_val = loss.item()`.
5. Return the tuple `(loss_val, logits)`. (Both must come from inside the `no_grad` block.)

Inputs:
- `model`: an `nn.Module`.
- `x, y`: same-shape input/target tensors.

**Critical behavior under no_grad.** Any tensor produced inside the block has `requires_grad=False` and `grad_fn=None`. Calling `.backward()` on `loss` (returned from inside the block) must raise `RuntimeError`. The test verifies this.

In [ ]:
def ex1_validate(model: t.nn.Module, x: Tensor, y: Tensor) -> tuple:
    """Validation pass under no_grad. Returns (loss_val, logits)."""
    raise NotImplementedError()


def _test_ex1():
    # A trivial linear model.
    model = t.nn.Linear(3, 2, bias=False)
    with t.no_grad():
        model.weight.copy_(t.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]))
    x = t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    y = t.tensor([[1.0, 2.0], [4.0, 5.0]])

    loss_val, logits = ex1_validate(model, x, y)

    assert isinstance(loss_val, float), (
        f'loss_val must be a Python float (from .item()), got {type(loss_val)}'
    )
    assert abs(loss_val) < 1e-6, (
        f'loss should be ~0 because logits exactly match y, got {loss_val}'
    )

    # logits must have been produced inside no_grad.
    assert logits.requires_grad is False, (
        'logits.requires_grad must be False — was the forward inside `with t.no_grad():` ?'
    )
    assert logits.grad_fn is None, (
        f'logits.grad_fn must be None inside no_grad; got {logits.grad_fn}'
    )

    # Backward on a tensor produced inside no_grad must raise.
    raised = False
    try:
        fake_loss = logits.sum()
        fake_loss.backward()
    except RuntimeError as e:
        raised = True
        assert 'grad' in str(e).lower() or 'requires_grad' in str(e).lower() or 'leaf' in str(e).lower(), (
            f'unexpected error: {e}'
        )
    assert raised, 'tensors from no_grad must not be backward-able'

    # Outside no_grad the model is still gradient-tracked.
    logits_train = model(x)
    assert logits_train.requires_grad is True, (
        'after exiting no_grad, model output must again require grad'
    )
    assert logits_train.grad_fn is not None
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_validate(model, x, y):
    with t.no_grad():
        logits = model(x)
        loss = ((logits - y) ** 2).mean()
        loss_val = loss.item()
        return loss_val, logits
```

**Why `no_grad` matters for memory.** Every op inside a gradient-tracked forward stashes the intermediate activations it needs to compute the backward. For a ResNet-50 forward at batch 32, that's hundreds of megabytes. `no_grad` skips the stashing entirely. For validation on a held-out test set this is a free 5-10× memory reduction.

**`no_grad` vs `inference_mode`.** `inference_mode` is PyTorch's stricter newer version. It additionally disables version counters and forbids any later `requires_grad=True` use of the produced tensors. Faster, but slightly less ergonomic — you can't accidentally re-enter training with a tensor that was made under it. ARENA uses both interchangeably.

**Not a substitute for `model.eval()`.** This drill targets only the autograd-control half of validation. The other half — switching BatchNorm/Dropout into eval mode — is a separate drill (`train-eval-mode-branch`). You need both for honest validation.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()